## 6. Analyse de l'influence des poids et préférences

On vérifie si en modifiant les poids ou les préférences le modèle se comporte comme attendu et l'impact sur le coût. 

In [1]:
import Pkg
Pkg.activate("..")

using JuMP
using HiGHS
include("../src/modele_v3.jl")
include("../src/donnees.jl")

data = charger_donnees();

  Activating project at `C:\Dev\workforce-scheduling-optimization`


In [2]:
function resoudre(data; p_min = 1, p_semaine = 1)
    model, x, e, viol_type, viol_jour, viol_semaine =
        construire_modele(data; p_min, p_semaine)
    set_silent(model)
    optimize!(model)
    assert_is_solved_and_feasible(model)
    return (; model, x, e, viol_type, viol_jour, viol_semaine)
end

resoudre (generic function with 1 method)

In [3]:
function violations_par_employe(r, data)
    # On part des affectations réelles et non des variables de violation :
    # à poids 0, viol_type et viol_jour peuvent valoir 1 sans violation réelle
    # round enlève les petites imprécisions du solveur (0.9999999 devient 1)
    affectations = round.(Int, value.(r.x))

    # Liste qui contiendra une ligne par employé concerné
    resultat = []

    # On regarde chaque employé, un par un
    for (p, i) in enumerate(data.I)
        nb_viol_type = 0
        nb_viol_jour = 0

        # Pour cet employé, on compte les shifts travaillés hors préférence
        for j in data.J
            if affectations[i, j] == 0
                continue
            end

            for (l, t) in enumerate(data.T)
                if data.PrefType[p, l] == 1 && t != data.TypeShift[j]
                    nb_viol_type = nb_viol_type + 1
                end
            end

            for (l, s) in enumerate(data.S)
                if data.PrefJour[p, l] == 1 && s != data.Jour[j]
                    nb_viol_jour = nb_viol_jour + 1
                end
            end
        end

        # On garde l'employé seulement s'il a au moins une violation
        if nb_viol_type + nb_viol_jour > 0
            ligne = (employe = i,
                     poids = data.PoidsPref[i],
                     type = nb_viol_type,
                     jour = nb_viol_jour)
            push!(resultat, ligne)
        end
    end

    return resultat
end

violations_par_employe (generic function with 1 method)

In [4]:
function indicateurs(r, data)
    # On part des affectations réelles : 1 si l'employé travaille le shift, 0 sinon
    # round enlève les petites imprécisions du solveur (0.9999999 devient 1)
    affectations = round.(Int, value.(r.x))

    # Compteurs de la solution
    violations_type = 0
    violations_jour = 0
    shifts_manquants = 0
    shifts_au_dela_de_5 = 0

    for (p, i) in enumerate(data.I)
        nb_shifts = 0

        for j in data.J
            # Le shift n'est pas travaillé : il n'y a rien à compter
            if affectations[i, j] == 0
                continue
            end

            nb_shifts += 1

            # Préférence de type : l'employé en a-t-il une, et ce shift la respecte-t-il ?
            for (l, t) in enumerate(data.T)
                if data.PrefType[p, l] == 1 && t != data.TypeShift[j]
                    violations_type += 1
                end
            end

            # Préférence de jour : même raisonnement
            for (l, s) in enumerate(data.S)
                if data.PrefJour[p, l] == 1 && s != data.Jour[j]
                    violations_jour += 1
                end
            end
        end

        # Écart au minimum souhaité et dépassement de la semaine chargée
        shifts_manquants += max(0, data.S_min[p] - nb_shifts)
        shifts_au_dela_de_5 += max(0, nb_shifts - 5)
    end

    return (shifts_manquants = shifts_manquants,
            violations_type = violations_type,
            violations_jour = violations_jour,
            shifts_au_dela_de_5 = shifts_au_dela_de_5,
            statut = termination_status(r.model),
            cout = objective_value(r.model))
end

indicateurs (generic function with 1 method)

On veut récupérer le nombre de violations de chaque type. On calcule d'abord le scénario de base. 

In [5]:
# Base
resultat_base = indicateurs(resoudre(data ; p_min = 1, p_semaine = 1), data)

(shifts_manquants = 4, violations_type = 1, violations_jour = 9, shifts_au_dela_de_5 = 2, statut = OPTIMAL, cout = 18.0)

In [6]:
# Modification des poids p_min, shifts minimums non atteints
resultat_p_min = indicateurs(resoudre(data ; p_min = 10, p_semaine = 1), data)

(shifts_manquants = 1, violations_type = 3, violations_jour = 10, shifts_au_dela_de_5 = 3, statut = OPTIMAL, cout = 30.0)

In [7]:
# Modification des poids p_semaine, semaines trop chargées
resultat_p_semaine = indicateurs(resoudre(data ; p_min = 1, p_semaine = 10), data)

(shifts_manquants = 6, violations_type = 1, violations_jour = 9, shifts_au_dela_de_5 = 0, statut = OPTIMAL, cout = 18.0)

In [8]:
# Changement de préférence de shift - Pas le poids, juste le type de shift
PT = copy(data.PrefType)
PT[5, :] = [1 0 0]                      # E05 préfère les matins
data_E05_matin = merge(data, (PrefType = PT,))
resultat_E05_matin = indicateurs(resoudre(data_E05_matin), data_E05_matin)

(shifts_manquants = 4, violations_type = 2, violations_jour = 9, shifts_au_dela_de_5 = 2, statut = OPTIMAL, cout = 19.0)

In [9]:
# Changement de préférence de jour - Pas le poids, juste le jour préféré
PJ = copy(data.PrefJour)
PJ[5, :] = [1 0 0 0 0 0 0]                      # E05 préfère les lundis aux samedis
data_E05_lundi = merge(data, (PrefJour = PJ,))
resultat_E05_lundi = indicateurs(resoudre(data_E05_lundi), data_E05_lundi)

(shifts_manquants = 4, violations_type = 1, violations_jour = 9, shifts_au_dela_de_5 = 2, statut = OPTIMAL, cout = 18.0)

In [10]:
# Suppression d'une préférence
PJ = copy(data.PrefJour)
PJ[5, :] = [0 0 0 0 0 0 0]                      # E05 n'a plus de préférence de jour
data_E05_supp = merge(data, (PrefJour = PJ,))
resultat_E05_supp = indicateurs(resoudre(data_E05_supp), data_E05_supp)

(shifts_manquants = 4, violations_type = 1, violations_jour = 6, shifts_au_dela_de_5 = 2, statut = OPTIMAL, cout = 15.0)

In [11]:
# Modification du poids d'une préférence
PP = copy(data.PoidsPref)
PP["E02"] = 0
PP["E07"] = 0
data_poids_pref = merge(data, (PoidsPref = PP,))
r_poids_pref = resoudre(data_poids_pref)
resultat_E02_E07_poids_pref = indicateurs(r_poids_pref, data_poids_pref)

(shifts_manquants = 4, violations_type = 4, violations_jour = 9, shifts_au_dela_de_5 = 2, statut = OPTIMAL, cout = 18.0)

In [12]:
# Modification du poids de préférences
# avec visualisation des employés impactés en conséquence
violations_par_employe(r_poids_pref,data_poids_pref)

6-element Vector{Any}:
 (employe = "E02", poids = 0, type = 2, jour = 0)
 (employe = "E03", poids = 3, type = 1, jour = 0)
 (employe = "E05", poids = 1, type = 0, jour = 3)
 (employe = "E07", poids = 0, type = 1, jour = 0)
 (employe = "E10", poids = 1, type = 0, jour = 4)
 (employe = "E17", poids = 1, type = 0, jour = 2)

En mettant à 0 les poids des employés 2 et 7, leurs préférences ne comptent plus. Le modèle se comporte comme attendu. 

In [13]:
# Solution initiale avec pénalités initiales
# pour avoir la répartition des contraintes violées 
r = resoudre(data)
violations_par_employe(r,data)

4-element Vector{Any}:
 (employe = "E03", poids = 3, type = 1, jour = 0)
 (employe = "E05", poids = 1, type = 0, jour = 3)
 (employe = "E10", poids = 1, type = 0, jour = 4)
 (employe = "E17", poids = 1, type = 0, jour = 2)

A l'issue de tous les tests, on trouve (les colonnes des violations sont des nombres bruts mais le coût est pondéré) :


| Scénario | Poids p_min | Poids p_semaine | Shifts <br> manquants | Violations <br> de type | Violations <br> de jour | Shifts <br> au-delà de 5 | Coût |
| --- | --- | --- | --- | --- | --- | --- | --- |
| Base | 1 | 1 | 4 | 1 | 9 | 2 | 18 |
| E05 préfère les matins | 1 | 1 | 4 | 2 | 9 | 2 | 19 |
| E05 préfère les lundis | 1 | 1 | 4 | 1 | 9 | 2 | 18 |
| E05 sans préférence de jour | 1 | 1 | 4 | 1 | 6 | 2 | 15 |
| Poids de E02 et E07 à 0 | 1 | 1 | 4 | 4 | 9 | 2 | 18 |
| Poids des shifts manquants à 10 | 10 | 1 | 1 | 3 | 10 | 3 | 30 |
| Poids des semaines chargées à 10 | 1 | 10 | 6 | 1 | 9 | 0 | 18 |

Si on pénalise fortement le fait de ne pas faire son minimum de shifts, on voit qu'il ne reste plus qu'un shift manquant. Si on pénalise fortement les semaines trop chargées, on arrive à avoir un planning optimal sans semaine chargée. On note qu'il y a plusieurs plannings optimaux. Par exemple si on pénalise fortement les semaines trop chargées, on a un coût à 18 comme pour le planning de base, le même nombre de violations de préférences que le planning de base mais pas de semaine surchargée. 

Le modèle minimise un coût total sans tenir compte de sa répartition entre les employés. Un employé à préférence de poids faible peut donc voir sa préférence non respectée plusieurs fois alors qu’un employé à préférence de poids plus élevé est peu touché. Dans le scénario de base, E10 (poids 1) a 4 violations, contre 1 pour E03 (poids 3). Pour une répartition plus équitable, on pourrait limiter le nombre de violations par employé ou pénaliser dans l’objectif le maximum de violations par employé.